# Day 24 — 12-1 momentum by hand

**Acceptance:** 12-1 momentum for a known trending stock matches a hand calculation.

12-1 momentum at session *t* is the return from *t − 252* to *t − 21* sessions:

$$\text{mom}_{12\text{-}1}(t) = \frac{p_{t-21}}{p_{t-252}} - 1$$

the last twelve months with the most recent one left out, because over a month
winners tend to give some back. *p* is `adj_close_eur`: the back-adjusted close in
euros, dividends reinvested.

The stock is **NVDA**, the obvious trending name of the period. The hand
calculation below does not use `finmgr.features`. It counts NVDA's stored bars back from
*t*, checks against `exchange_calendars` that those bars are exactly XNAS's sessions
(so a row is a session), and divides two numbers. Then it asks
`read_momentum` for the same value.

Run headless with `make notebooks`. It needs the local store (`finmgr ingest`, `finmgr fx`).

In [1]:
import math

import exchange_calendars as xcals
import pandas as pd

from finmgr.data.benchmarks import read_total_return

TICKER = "NVDA"
AS_OF = "2026-10-14"  # pinned so the numbers below are reproducible
t = pd.Timestamp(AS_OF).date()

bars = read_total_return(TICKER, as_of=AS_OF, start="2024-06-01")
bars = bars.sort_values("date").reset_index(drop=True)
print(len(bars), "bars,", bars["date"].min(), "to", bars["date"].max())

594 bars, 2024-06-03 to 2026-10-14


## Step 1 — count back 21 and 252 sessions

Position *i* of *t* in the bar list, then rows *i − 21* and *i − 252*.

In [2]:
i = bars.index[bars["date"] == t][0]
picked = bars.loc[[i - 252, i - 21, i], ["date", "adj_close_eur"]]
picked.index = ["t-252", "t-21", "t"]
picked

,date,adj_close_eur
t-252,2025-10-13,161.923186
t-21,2026-09-15,183.720125
t,2026-10-14,197.510243


## Step 2 — check rows are sessions

If a bar were missing, counting rows would reach further back than 252 sessions.
Compare the stored dates in the window with XNAS's own session list.

In [3]:
cal = xcals.get_calendar("XNAS")
window = bars.loc[i - 252 : i, "date"]
xnas = [s.date() for s in cal.sessions_in_range(window.iloc[0], window.iloc[-1])]
print("stored bars in window:", len(window), "| XNAS sessions in window:", len(xnas))
assert list(window) == xnas, "store and calendar disagree; row counting would be wrong"
print("every session from t-252 to t has exactly one bar")

stored bars in window: 253 | XNAS sessions in window: 253
every session from t-252 to t has exactly one bar


## Step 3 — the hand calculation

In [4]:
p_252 = picked.loc["t-252", "adj_close_eur"]
p_21 = picked.loc["t-21", "adj_close_eur"]
p_t = picked.loc["t", "adj_close_eur"]

hand_12_1 = p_21 / p_252 - 1
hand_12m = p_t / p_252 - 1  # for contrast: the year *including* the last month
print(f"p[t-252] = {p_252:.6f}  p[t-21] = {p_21:.6f}  p[t] = {p_t:.6f}")
print(f"12-1 momentum by hand : {hand_12_1:+.6%}")
print(f"12-month return       : {hand_12m:+.6%}   (last month alone: {p_t / p_21 - 1:+.4%})")

p[t-252] = 161.923186  p[t-21] = 183.720125  p[t] = 197.510243
12-1 momentum by hand : +13.461283%
12-month return       : +21.977740%   (last month alone: +7.5060%)


## Step 4 — the module, and the comparison

In [5]:
from finmgr.features.momentum import read_momentum

mom = read_momentum(TICKER, as_of=AS_OF, start=AS_OF)
module_12_1 = mom.loc[mom["date"] == t, "mom_12_1"].item()
print(f"read_momentum mom_12_1: {module_12_1:+.6%}")
print(f"hand calculation      : {hand_12_1:+.6%}")
print(f"difference            : {module_12_1 - hand_12_1:.2e}")
assert math.isclose(module_12_1, hand_12_1, rel_tol=1e-12)
assert hand_12_1 > 0, "NVDA was picked as a trending stock; 12-1 should be positive"
print("MATCH")

read_momentum mom_12_1: +13.461283%
hand calculation      : +13.461283%
difference            : 0.00e+00
MATCH


## Step 5 — not just one date

The same hand rule at every month-end of the last year, against the module.

In [6]:
full = read_momentum(TICKER, as_of=AS_OF, start="2025-10-01").set_index("date")
rows = []
for end in pd.date_range("2025-10-01", AS_OF, freq="BME"):
    day = bars.loc[bars["date"] <= end.date(), "date"].iloc[-1]
    j = bars.index[bars["date"] == day][0]
    hand = bars.loc[j - 21, "adj_close_eur"] / bars.loc[j - 252, "adj_close_eur"] - 1
    rows.append({"date": day, "hand": hand, "module": full.loc[day, "mom_12_1"]})
check = pd.DataFrame(rows).set_index("date")
check["abs_diff"] = (check["hand"] - check["module"]).abs()
assert check["abs_diff"].max() < 1e-12
print(f"{len(check)} month-ends, max |hand - module| = {check['abs_diff'].max():.1e}")
print(
    check.to_string(
        formatters={
            "hand": "{:+.4%}".format,
            "module": "{:+.4%}".format,
            "abs_diff": "{:.1e}".format,
        }
    )
)

12 month-ends, max |hand - module| = 0.0e+00
                hand    module abs_diff
date                                   
2025-10-31 +23.3089% +23.3089%  0.0e+00
2025-11-28 +36.8715% +36.8715%  0.0e+00
2025-12-31 +18.0074% +18.0074%  0.0e+00
2026-01-30 +34.3399% +34.3399%  0.0e+00
2026-02-27 +27.7909% +27.7909%  0.0e+00
2026-03-31 +52.8563% +52.8563%  0.0e+00
2026-04-30 +59.2982% +59.2982%  0.0e+00
2026-05-29 +50.2105% +50.2105%  0.0e+00
2026-06-30 +34.3193% +34.3193%  0.0e+00
2026-07-31 +11.6913% +11.6913%  0.0e+00
2026-08-31 +12.7757% +12.7757%  0.0e+00
2026-09-30 +22.8137% +22.8137%  0.0e+00
